# LoRA Fine-Tuning: Mistral-7B-Instruct for Financial Statement Analysis

Fine-tunes `unsloth/mistral-7b-instruct-v0.3-bnb-4bit` with LoRA (via [Unsloth](https://github.com/unslothai/unsloth)) to analyze structured financial statement data (extracted and normalized in earlier pipeline steps) and produce a structured JSON output: computed financial ratios, an overall assessment, strengths/weaknesses, and recommendations.

Run on Google Colab (single GPU, 4-bit quantization).

## 1. Setup
Install Unsloth and dependencies for fast LoRA fine-tuning on a single Colab GPU.

In [ ]:
# ───────────────────────────
# 1. Installation
# ───────────────────────────
!pip install -q unsloth transformers datasets trl accelerate bitsandbytes

## 2. Load base model + attach LoRA adapters
Base model: `unsloth/mistral-7b-instruct-v0.3-bnb-4bit` (4-bit quantized Mistral-7B-Instruct).
LoRA is applied to all attention and MLP projection layers (`q/k/v/o_proj`, `gate/up/down_proj`) with `r=16`, `alpha=64`.

In [ ]:
# ─────────────────────────────
# 2. Imports
# ─────────────────────────────
from unsloth import FastLanguageModel
import torch
import json
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import os
# ─────────────────────────────
# 3. Charger modèle (HuggingFace)
# ─────────────────────────────
model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=4096,   # ← 1024 trop petit, output JSON dépasse facilement
    load_in_4bit=True,
    dtype=None,
)

# ─────────────────────────────
# 4. LoRA
# ─────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                  # ← 8 → 16 : meilleure capacité d'apprentissage
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],  # ← couvre tout le modèle
    lora_alpha=64,
    lora_dropout=0.15,
    bias="none",
    use_gradient_checkpointing="unsloth",
)



## 3. Load & format the fine-tuning dataset
Loads the labeled JSON dataset (built from `2_data_prep/`) and formats each example into Mistral's native `[INST] ... [/INST]` instruction format, with the model's completion being the structured JSON financial analysis.

In [ ]:
# ─────────────────────────────
# 5. Charger dataset
# ─────────────────────────────
dataset = load_dataset(
    "json",
    data_files="/content/clean_finetuning_dataset_v4_qlora_clean.json",  # ← chemin exact
    split="train"
)

print(f"✅ Dataset chargé : {len(dataset)} exemples")

# ─────────────────────────────
# 6. Format prompt
# ─────────────────────────────
def format_prompt(example):
    instruction = example.get("instruction", "")
    financial_data = example.get("input", {})
    output = example.get("output", {})

    user_content = (
        f"{instruction}\n\n"
        f"Source: {financial_data.get('source_file', 'N/A')}\n"
        f"Period: {financial_data.get('period', 'N/A')}\n"
        f"Unit: {financial_data.get('reporting_unit', 'TND')}\n"
        f"Financial Data: {json.dumps(financial_data.get('financial_data', {}), ensure_ascii=False)}"
    )

    completion = json.dumps(output, ensure_ascii=False, indent=2)

    # Format natif Mistral Instruct
    text = (
        f"<s>[INST] {user_content} [/INST]\n"
        f"{completion}</s>"
    )

    return {"text": text}

dataset = dataset.map(format_prompt, remove_columns=list(dataset.column_names))

## 4. Check sequence lengths
Before training, tokenize the full dataset to check the token-length distribution against the 4096-token context limit, to catch truncation risk early.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Tokeniser tout le dataset
lengths = []
for example in dataset:
    tokens = tokenizer(example["text"], return_tensors="pt")
    lengths.append(tokens["input_ids"].shape[1])

lengths = np.array(lengths)

# ── Statistiques ──────────────────────────────
print(f"Nombre d'exemples   : {len(lengths)}")
print(f"Min                 : {lengths.min()} tokens")
print(f"Max                 : {lengths.max()} tokens")
print(f"Moyenne             : {lengths.mean():.0f} tokens")
print(f"Médiane             : {np.median(lengths):.0f} tokens")
print(f"95e percentile      : {np.percentile(lengths, 95):.0f} tokens")
print(f"99e percentile      : {np.percentile(lengths, 99):.0f} tokens")
print(f"\n⚠️  Exemples > 4096 tokens : {(lengths > 4096).sum()} ({(lengths > 4096).mean()*100:.1f}%)")

# ── Histogramme ───────────────────────────────
plt.figure(figsize=(10, 4))
plt.hist(lengths, bins=50, color="steelblue", edgecolor="white")
plt.axvline(4096, color="red", linestyle="--", label="Limite 4096")
plt.xlabel("Longueur (tokens)")
plt.ylabel("Nombre d'exemples")
plt.title("Distribution des longueurs de séquences")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Train
90/10 train/test split, then supervised fine-tuning (SFT) with the LoRA adapters. Uses cosine LR schedule, 8-bit AdamW, and gradient checkpointing to fit on a single Colab GPU.

In [ ]:
# ─────────────────────────────
# 7. Training
# ─────────────────────────────
#-----------------Splitting the data into training and test---------------------
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
print(f"Train : {len(dataset_split['train'])} exemples")
print(f"Test  : {len(dataset_split['test'])} exemples")
#-------------------------------------------------------------------------------
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_split["train"],
    eval_dataset=dataset_split["test"],
    dataset_text_field="text",
    max_seq_length=4096,
    packing=True,
    dataset_num_proc=2,
    args=TrainingArguments(
        output_dir="./mistral-7b-instruct-finetuned",
        num_train_epochs=5,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=0.00014276523636006766,
        lr_scheduler_type="cosine",
        warmup_ratio=0.06,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        logging_steps=5,
        save_steps=20,
        eval_strategy="steps",
        eval_steps=20,
        load_best_model_at_end=True,
        seed=42,
    ),
)

trainer.train()

## 6. Training curves
Plot train vs. eval loss over steps, and the eval-minus-train loss gap as a simple overfitting check.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ── Extraire les logs ──────────────────────────────────────────────
history = trainer.state.log_history

train_logs = [e for e in history if "loss" in e and "eval_loss" not in e]
eval_logs  = [e for e in history if "eval_loss" in e]

train_steps  = [e["step"] for e in train_logs]
train_losses = [e["loss"] for e in train_logs]
eval_steps   = [e["step"] for e in eval_logs]
eval_losses  = [e["eval_loss"] for e in eval_logs]

# ── Figure ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f"r={model.peft_config['default'].r}  "
             f"α={model.peft_config['default'].lora_alpha}  "
             f"lr={trainer.args.learning_rate}  "
             f"dropout={model.peft_config['default'].lora_dropout}",
             fontsize=12)

# 1. Courbes de loss
ax = axes[0]
ax.plot(train_steps, train_losses, label="train loss", color="steelblue")
ax.plot(eval_steps,  eval_losses,  label="eval loss",  color="tomato", linewidth=2)
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Courbes d'apprentissage")
ax.legend()
ax.grid(alpha=0.3)

# 2. Écart train/eval (signe d'overfitting)
ax = axes[1]
# Aligner les steps eval avec les valeurs train les plus proches
gaps = []
for i, es in enumerate(eval_steps):
    closest = min(range(len(train_steps)), key=lambda j: abs(train_steps[j]-es))
    gaps.append(eval_losses[i] - train_losses[closest])
ax.bar(eval_steps, gaps, color=["tomato" if g > 0.05 else "steelblue" for g in gaps])
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_xlabel("Step")
ax.set_ylabel("eval_loss − train_loss")
ax.set_title("Écart overfitting")
ax.grid(axis="y", alpha=0.3)

# 3. Vitesse de convergence (dérivée de eval_loss)
ax = axes[2]
if len(eval_losses) > 1:
    deltas = [eval_losses[i] - eval_losses[i-1] for i in range(1, len(eval_losses))]
    ax.bar(eval_steps[1:], deltas,
           color=["#2ecc71" if d < 0 else "tomato" for d in deltas])
    ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
    ax.set_xlabel("Step")
    ax.set_ylabel("Δ eval_loss")
    ax.set_title("Progression par step (↓ = amélioration)")
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("training_viz.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Meilleure eval_loss : {min(eval_losses):.4f} au step {eval_steps[eval_losses.index(min(eval_losses))]}")

In [ ]:
import matplotlib.pyplot as plt

# Après trainer.train()
logs = trainer.state.log_history

train_loss = [(l["step"], l["loss"]) for l in logs if "loss" in l]
eval_loss  = [(l["step"], l["eval_loss"]) for l in logs if "eval_loss" in l]

steps_t, loss_t = zip(*train_loss)
steps_e, loss_e = zip(*eval_loss)

plt.figure(figsize=(10, 4))
plt.plot(steps_t, loss_t, label="Train loss", color="steelblue")
plt.plot(steps_e, loss_e, label="Eval loss",  color="tomato", linestyle="--")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Train vs Eval Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Perplexity
Convert loss to perplexity (`e^loss`) as a more interpretable measure of how confidently the model predicts the next token.

In [ ]:
import numpy as np

# Perplexité = e^loss  → mesure combien de tokens le modèle "hésite"
perplexity_train = [np.exp(l) for l in loss_t]
perplexity_eval  = [np.exp(l) for l in loss_e]

plt.figure(figsize=(10, 4))
plt.plot(steps_t, perplexity_train, label="Train perplexity", color="steelblue", linewidth=2)
plt.plot(steps_e, perplexity_eval,  label="Eval perplexity",  color="tomato", linestyle="--", linewidth=2)
plt.axhline(y=1.0, color="green", linestyle=":", label="Perplexité parfaite = 1")
plt.xlabel("Steps")
plt.ylabel("Perplexité")
plt.title("Perplexité au cours du training")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Perplexité finale train : {perplexity_train[-1]:.2f}")
print(f"Perplexité finale eval  : {perplexity_eval[-1]:.2f}")
# Interprétation : perplexité = 1.5 → le modèle hésitokenste entre ~1.5  à chaque position

## 8. Evaluate output validity
Run the fine-tuned model on the held-out test set and check what fraction of generations are valid, parseable JSON.

In [ ]:
valid_json = 0
total = len(dataset_split["test"])

for example in dataset_split["test"]:
    text = example["text"]
    prompt = text[:text.index("[/INST]") + len("[/INST]")]
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = generated[len(prompt):]

    try:
        json.loads(response.strip())
        valid_json += 1
    except:
        pass

print(f"\n📊 JSON valides : {valid_json}/{total} ({valid_json/total*100:.0f}%)")

In [ ]:
results = []

for i, example in enumerate(dataset_split["test"]):
    text = example["text"]
    prompt = text[:text.index("[/INST]") + len("[/INST]")]
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=1024,
            temperature=0.1, do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt):]

    try:
        json.loads(response.strip())
        results.append({"exemple": i+1, "status": "✅ Valide"})
    except:
        results.append({"exemple": i+1, "status": "❌ Invalide"})

# Affichage
colors = ["green" if r["status"].startswith("✅") else "tomato" for r in results]
plt.figure(figsize=(8, 3))
plt.bar([r["exemple"] for r in results], [1]*len(results), color=colors)
plt.xlabel("Exemple test")
plt.title("Validité JSON par exemple")
plt.yticks([])
from matplotlib.patches import Patch
plt.legend(handles=[Patch(color="green", label="JSON valide"),
                    Patch(color="tomato", label="JSON invalide")])
plt.tight_layout()
plt.show()

## 9. Save the fine-tuned adapter

In [ ]:
# ─────────────────────────────
# 8. Sauvegarde
# ─────────────────────────────
model.save_pretrained("./mistral-7b-instruct-finetuned-lora")
tokenizer.save_pretrained("./mistral-7b-instruct-finetuned-lora")
print("✅ Fine-tuning terminé !")

## 10. Export to GGUF (optional)
Quantize and export to GGUF format so the fine-tuned model can be served locally via Ollama.

In [ ]:
# ─────────────────────────────
# Sauvegarder en format GGUF pour Ollama (recommandé)
# ─────────────────────────────
model.save_pretrained_gguf(
    "mistral-finetuned-gguf",
    tokenizer,
    quantization_method="q4_k_m"   # bon équilibre qualité/taille
)

# ─────────────────────────────
# Télécharger depuis Colab
# ─────────────────────────────
from google.colab import files
import os, glob

for f in glob.glob("mistral-finetuned-gguf/*.gguf"):
    files.download(f)

## 11. Inference
Helper functions to normalize a raw uniformized financial-statement JSON into the model's expected input format, and run inference.

In [ ]:
# ─────────────────────────────
# 9. Inférence
# ─────────────────────────────
FastLanguageModel.for_inference(model)

def normalize_input(raw_data):
    if "statement_data" in raw_data:
        latest_year = max(raw_data["statement_data"].keys())
        flat = raw_data["statement_data"][latest_year]
        return {
            "source_file": raw_data.get("source_file"),
            "period": latest_year,
            "reporting_unit": raw_data.get("reporting_unit", "TND"),
            "financial_data": flat
        }
    return raw_data

def predict(instruction, financial_data):
    user_content = (
        f"{instruction}\n\n"
        f"Source: {financial_data.get('source_file', 'N/A')}\n"
        f"Period: {financial_data.get('period', 'N/A')}\n"
        f"Unit: {financial_data.get('reporting_unit', 'TND')}\n"
        f"Financial Data: {json.dumps(financial_data.get('financial_data', {}), ensure_ascii=False)}"
    )

    # Même format que l'entraînement
    prompt = f"<s>[INST] {user_content} [/INST]\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=1500,   # ← assez grand pour le JSON complet
        temperature=0.1,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = decoded.split("[/INST]")[-1].strip()
    return response

In [ ]:
# Test
raw = {
    "source_file": "output_lila.xlsx",
    "period": "2025",
    "reporting_unit": "TND",
    "financial_data": {
        "cash": None,
        "current_assets": 350067345.0,
        "current_liabilities": 377498072.0,
        "equity": 307185389.0,
        "financial_expense_net": -10300090.0,
        "net_income": 22302543.0,
        "non_current_assets": 364291573.0,
        "non_current_liabilities": 29675457.0,
        "receivables_net": None,
        "revenue": 220500318.0,
        "short_term_investments": 0.0,
        "stocks_net": None,
        "total_assets": 714358918.0,
        "total_liabilities": 407173529.0,
        "total_operating_expense": 190747848.0,
        "total_operating_income": 222383702.0,
        "operating_result": 31635854.0
    }
}

result = predict(
    instruction="Analyze the structured financial data below. Compute the listed ratios only when they are economically meaningful, and provide concise English interpretations grounded strictly in the provided values.",
    financial_data=normalize_input(raw)
)

print(result)

## 12. Render results as a dashboard
Parse the model's JSON output (ratios, strengths/weaknesses, recommendations) and render it as an HTML financial-analysis dashboard for quick visual review.

In [ ]:
# ─────────────────────────────────────────────────────
# FIX CELL — replace parse_model_output + predict call
# ─────────────────────────────────────────────────────

import json, re
from IPython.display import display, HTML

def parse_model_output(raw_text: str) -> dict:
    """
    Robust extraction of the model's JSON response.
    Handles: prompt text before the JSON, markdown fences, truncation.
    """
    # Step 1 — isolate everything AFTER [/INST]
    if "[/INST]" in raw_text:
        raw_text = raw_text.split("[/INST]")[-1]

    text = raw_text.strip()

    # Step 2 — strip markdown fences if present
    if "```" in text:
        for part in text.split("```"):
            part = part.strip().lstrip("json").strip()
            if part.startswith("{"):
                text = part
                break

    # Step 3 — find the LAST top-level '{' … '}' block
    # (the model's output, not the financial data dict in the prompt)
    brace_start = text.rfind('\n{')          # model output always starts on a new line
    if brace_start == -1:
        brace_start = text.rfind('{')        # fallback
    text = text[brace_start:].strip()

    # Step 4 — if truncated, close all open braces/brackets
    open_braces   = text.count('{') - text.count('}')
    open_brackets = text.count('[') - text.count(']')
    if open_braces > 0 or open_brackets > 0:
        # Remove trailing incomplete key-value pair (e.g. `"revenue": {`)
        text = re.sub(r',?\s*"[^"]*"\s*:\s*\{?\s*$', '', text.rstrip())
        text += ']' * max(0, open_brackets)
        text += '}' * max(0, open_braces)

    return json.loads(text)


# ── Increase max_new_tokens so JSON is never cut off ──
def predict_fixed(instruction, financial_data):
    user_content = (
        f"{instruction}\n\n"
        f"Source: {financial_data.get('source_file', 'N/A')}\n"
        f"Period: {financial_data.get('period', 'N/A')}\n"
        f"Unit: {financial_data.get('reporting_unit', 'TND')}\n"
        f"Financial Data: {json.dumps(financial_data.get('financial_data', {}), ensure_ascii=False)}"
    )
    prompt = f"<s>[INST] {user_content} [/INST]\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=2500,       # ← was 1500, JSON easily exceeds that
        temperature=0.1,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("[/INST]")[-1].strip()


# ─────────────────────────────────────────────────────
# Rating helpers
# ─────────────────────────────────────────────────────
RATING_STYLE = {
    "STRONG":   ("#EAF3DE", "#27500A"),
    "GOOD":     ("#E6F1FB", "#0C447C"),
    "MODERATE": ("#FAEEDA", "#633806"),
    "WEAK":     ("#FAECE7", "#712B13"),
    "CRITICAL": ("#FCEBEB", "#791F1F"),
    "ADEQUATE": ("#E1F5EE", "#085041"),
    "HIGH":     ("#FCEBEB", "#791F1F"),
}
BAR_COLOR = {
    "STRONG": "#639922", "GOOD": "#378ADD", "MODERATE": "#EF9F27",
    "WEAK": "#D85A30",   "CRITICAL": "#E24B4A", "ADEQUATE": "#1D9E75",
    "HIGH": "#E24B4A",
}
def bar_w(value, unit):
    if unit == "%":  return min(100, abs(float(value)))
    if unit == "x":  return min(100, abs(float(value)) * 25)
    return 50

def ratio_card(r):
    rating = str(r.get("rating","MODERATE")).upper()
    bg, fg = RATING_STYLE.get(rating, ("#eee","#333"))
    bc     = BAR_COLOR.get(rating, "#888")
    bw     = bar_w(r.get("value",0), r.get("unit",""))
    return f"""
    <div style="background:#fff;border:0.5px solid #e0dfd8;border-radius:12px;
                padding:14px 16px;display:flex;flex-direction:column;gap:6px">
      <div style="display:flex;justify-content:space-between;align-items:center;gap:8px">
        <span style="font-size:12px;color:#6b6a64;flex:1">{r.get('name','')}</span>
        <span style="font-size:11px;font-weight:500;padding:3px 10px;border-radius:20px;
                     white-space:nowrap;background:{bg};color:{fg}">{rating}</span>
      </div>
      <div style="font-size:22px;font-weight:500;color:#1a1a18">
        {r.get('value','N/A')} {r.get('unit','')}
      </div>
      <div style="height:6px;background:#f0efe8;border-radius:3px;overflow:hidden">
        <div style="height:100%;width:{bw:.0f}%;background:{bc};border-radius:3px"></div>
      </div>
      <div style="font-size:11px;color:#8a8880">{r.get('formula','')}</div>
      <div style="font-size:12px;color:#6b6a64">{r.get('interpretation','')}</div>
    </div>"""

def section(title, cols, keys, ratios):
    cards = "".join(ratio_card(ratios[k]) for k in keys if k in ratios)
    if not cards: return ""
    return f"""
    <div style="margin-bottom:24px">
      <div style="font-size:11px;font-weight:500;color:#8a8880;letter-spacing:.07em;
                  text-transform:uppercase;margin-bottom:10px">{title}</div>
      <div style="display:grid;grid-template-columns:repeat({cols},minmax(0,1fr));gap:10px">
        {cards}
      </div>
    </div>"""

def render(raw_text, meta):
    try:
        data = parse_model_output(raw_text)
    except Exception as e:
        display(HTML(f"<pre style='color:red'>Parse error: {e}\n\n{raw_text[:800]}</pre>"))
        return

    ratios     = data.get("ratios", {})
    conclusion = data.get("conclusion", {})
    overall    = conclusion.get("overall_assessment","N/A").upper()
    ov_bg,ov_fg = RATING_STYLE.get(overall,("#eee","#333"))

    src    = meta.get("source_file","N/A")
    period = meta.get("period","N/A")
    unit   = meta.get("reporting_unit","TND").upper()

    # ── Key metrics strip ─────────────────────────────
    fd = meta.get("financial_data", {})
    def fmt(v): return f"{v/1e6:.1f}M" if v and abs(v)>=1e6 else (str(v) if v else "N/A")
    metrics = [
        ("Revenue",      fd.get("revenue")),
        ("Net income",   fd.get("net_income")),
        ("Total assets", fd.get("total_assets")),
        ("Equity",       fd.get("equity")),
    ]
    metric_html = "".join(f"""
        <div style="background:#f7f7f5;border-radius:8px;padding:12px 14px">
          <div style="font-size:12px;color:#8a8880;margin-bottom:4px">{label}</div>
          <div style="font-size:16px;font-weight:500;color:#1a1a18">{fmt(val)}</div>
          <div style="font-size:11px;color:#b0afa8">{unit}</div>
        </div>""" for label, val in metrics)

    # ── Strengths / Weaknesses ────────────────────────
    def ul(items, color):
        li = "".join(f"<li style='margin-bottom:3px'>{i}</li>" for i in items)
        return f"<ul style='margin:0;padding-left:16px;font-size:13px;color:#1a1a18;line-height:1.8'>{li}</ul>"

    sw = f"""
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:12px">
      <div style="background:#f0f9f4;border:0.5px solid #a5d6b8;border-radius:12px;padding:14px 16px">
        <div style="font-size:11px;font-weight:500;color:#27500A;letter-spacing:.05em;
                    text-transform:uppercase;margin-bottom:8px">Strengths</div>
        {ul(conclusion.get('strengths',[]), '#27500A')}
      </div>
      <div style="background:#fef2f2;border:0.5px solid #f7c1c1;border-radius:12px;padding:14px 16px">
        <div style="font-size:11px;font-weight:500;color:#791F1F;letter-spacing:.05em;
                    text-transform:uppercase;margin-bottom:8px">Weaknesses</div>
        {ul(conclusion.get('weaknesses',[]), '#791F1F')}
      </div>
    </div>"""

    recs = conclusion.get("recommendations", [])
    rec_html = f"""
    <div style="background:#fffbf0;border:0.5px solid #FAC775;border-radius:12px;
                padding:14px 16px;margin-bottom:12px">
      <div style="font-size:11px;font-weight:500;color:#633806;letter-spacing:.05em;
                  text-transform:uppercase;margin-bottom:8px">Recommendations</div>
      {ul(recs, '#633806')}
    </div>""" if recs else ""

    summary = conclusion.get("summary_paragraph","")
    sum_html = f"""
    <div style="background:#f7f7f5;border-radius:12px;padding:14px 16px;
                font-size:13px;color:#6b6a64;line-height:1.7">{summary}</div>
    """ if summary else ""

    html = f"""
    <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
                max-width:900px;padding:8px 0">

      <div style="display:flex;align-items:baseline;flex-wrap:wrap;gap:10px;margin-bottom:16px">
        <span style="font-size:18px;font-weight:500;color:#1a1a18">
          {src.replace('.xlsx','').replace('_',' ')}
        </span>
        <span style="font-size:13px;color:#8a8880">{period} · {unit}</span>
        <span style="margin-left:auto;font-size:11px;font-weight:500;padding:4px 14px;
                     border-radius:20px;background:{ov_bg};color:{ov_fg}">
          Overall: {overall}
        </span>
      </div>

      <div style="display:grid;grid-template-columns:repeat(4,minmax(0,1fr));
                  gap:10px;margin-bottom:24px">
        {metric_html}
      </div>

      {section("Profitability", 2,
          ["operating_profit_margin","net_profit_margin",
           "return_on_assets","roa","return_on_equity","roe"], ratios)}
      {section("Liquidity", 3,
          ["current_ratio","quick_ratio","cash_ratio"], ratios)}
      {section("Leverage", 3,
          ["debt_to_assets","debt_to_equity","interest_coverage"], ratios)}

      <div style="font-size:11px;font-weight:500;color:#8a8880;letter-spacing:.07em;
                  text-transform:uppercase;margin-bottom:10px">Conclusion</div>
      {sw}
      {rec_html}
      {sum_html}
    </div>"""

    display(HTML(html))


# ─────────────────────────────────────────────────────
# Run
# ─────────────────────────────────────────────────────
raw = {
    "source_file": "output_lila.xlsx",
    "period": "2025",
    "reporting_unit": "TND",
    "financial_data": {
        "cash": None,
        "current_assets": 350067345.0,
        "current_liabilities": 377498072.0,
        "equity": 307185389.0,
        "financial_expense_net": -10300090.0,
        "net_income": 22302543.0,
        "non_current_assets": 364291573.0,
        "non_current_liabilities": 29675457.0,
        "receivables_net": None,
        "revenue": 220500318.0,
        "short_term_investments": 0.0,
        "stocks_net": None,
        "total_assets": 714358918.0,
        "total_liabilities": 407173529.0,
        "total_operating_expense": 190747848.0,
        "total_operating_income": 222383702.0,
        "operating_result": 31635854.0,
    }
}

normed = normalize_input(raw)
result = predict_fixed(
    instruction="Analyze the structured financial data below. Compute the listed ratios only when they are economically meaningful, and provide concise English interpretations grounded strictly in the provided values.",
    financial_data=normed
)

render(result, {**normed, "financial_data": raw["financial_data"]})